In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -------------------------
# Read from Silver (batch semantics)
# -------------------------
def _silver_obs():
    # Use dlt.read (not read_stream) so Gold is computed as a batch snapshot per run (Triggered mode)
    return dlt.read("canada_business.silver.silver_fred_observations")

def _silver_runs():
    return dlt.read("canada_business.silver.silver_fred_runs")


# ------------------------------------------------
# GOLD 0: Daily fact (BI-friendly copy of Silver)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_daily",
    comment="Daily FRED observations (clean, BI-ready). One row per series_id per day."
)
def gold_fred_daily():
    df = _silver_obs()

    return (
        df.select(
            "series_id",
            F.col("observation_date").cast("date").alias("observation_date"),
            F.col("value_double").cast("double").alias("value"),
            F.col("value_raw").cast("string").alias("value_raw"),
            F.col("_ingest_ts").alias("ingest_ts"),
            F.col("_source_file").alias("source_file"),
            F.col("realtime_start").cast("date").alias("realtime_start"),
            F.col("realtime_end").cast("date").alias("realtime_end"),
            F.col("obs_realtime_start").cast("date").alias("obs_realtime_start"),
            F.col("obs_realtime_end").cast("date").alias("obs_realtime_end"),
        )
        .where(F.col("series_id").isNotNull() & F.col("observation_date").isNotNull())
    )


# ------------------------------------------------
# GOLD 1: Latest value per series (dashboard KPI)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_latest",
    comment="Latest available value per series (for KPI tiles / overview dashboards)."
)
def gold_fred_latest():
    df = dlt.read("gold_fred_daily")

    # Get latest date per series
    latest_dates = (
        df.groupBy("series_id")
          .agg(F.max("observation_date").alias("latest_date"))
    )

    # Join back to get the corresponding row
    latest = (
        df.join(latest_dates, on="series_id", how="inner")
          .where(F.col("observation_date") == F.col("latest_date"))
          .drop("latest_date")
    )

    return latest


# ------------------------------------------------
# GOLD 2: Monthly aggregates (BI time grain)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_monthly",
    comment="Monthly aggregates per series (avg/min/max/last)."
)
def gold_fred_monthly():
    df = dlt.read("gold_fred_daily")

    m = df.withColumn("year_month", F.date_trunc("month", F.col("observation_date")).cast("date"))

    # "last_value_in_month" using max_by on observation_date (Spark supports max_by)
    return (
        m.groupBy("series_id", "year_month")
         .agg(
             F.avg("value").alias("avg_value"),
             F.min("value").alias("min_value"),
             F.max("value").alias("max_value"),
             F.max_by(F.col("value"), F.col("observation_date")).alias("last_value_in_month"),
             F.count("*").alias("days_in_month")
         )
    )


# ------------------------------------------------
# GOLD 3: YoY % change (on monthly avg)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_yoy",
    comment="Year-over-year % change computed from monthly avg_value."
)
def gold_fred_yoy():
    df = dlt.read("gold_fred_monthly")

    w = Window.partitionBy("series_id").orderBy(F.col("year_month"))

    return (
        df.withColumn("avg_value_prev_year", F.lag("avg_value", 12).over(w))
          .withColumn(
              "yoy_pct",
              F.when(F.col("avg_value_prev_year").isNull(), F.lit(None).cast("double"))
               .when(F.col("avg_value_prev_year") == 0, F.lit(None).cast("double"))
               .otherwise((F.col("avg_value") - F.col("avg_value_prev_year")) / F.col("avg_value_prev_year") * 100.0)
          )
          .select("series_id", "year_month", "avg_value", "avg_value_prev_year", "yoy_pct")
    )


# ------------------------------------------------
# GOLD 4: Rolling 12-month average (monthly)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_rolling_12m",
    comment="Rolling 12-month average of monthly avg_value (smooth trend)."
)
def gold_fred_rolling_12m():
    df = dlt.read("gold_fred_monthly")

    w = (
        Window.partitionBy("series_id")
              .orderBy(F.col("year_month"))
              .rowsBetween(-11, 0)
    )

    return (
        df.withColumn("rolling_12m_avg", F.avg("avg_value").over(w))
          .select("series_id", "year_month", "avg_value", "rolling_12m_avg")
    )


# ------------------------------------------------
# GOLD 5: 30-day volatility (daily stddev)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_volatility_30d",
    comment="30-day rolling volatility (stddev) on daily values per series."
)
def gold_fred_volatility_30d():
    df = dlt.read("gold_fred_daily")

    w = (
        Window.partitionBy("series_id")
              .orderBy(F.col("observation_date"))
              .rowsBetween(-29, 0)
    )

    return (
        df.select("series_id", "observation_date", "value")
          .withColumn("volatility_30d", F.stddev_samp("value").over(w))
    )


# ------------------------------------------------
# GOLD 6: Run / audit table (carry forward)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_runs",
    comment="Gold copy of run audit derived from manifest (good for lineage & dashboards)."
)
def gold_fred_runs():
    return _silver_runs()

In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def week_ending_sunday(dcol):
    return F.next_day(F.date_sub(dcol, 1), "Sun")

@dlt.table(
  name="gold_macro_demand_weekly",
  comment="Weekly macro + demand with forward-filled macro values"
)
def gold_macro_demand_weekly():

    fred = dlt.read("canada_business.silver.silver_fred_observations")

    series_map = {
        "DCOILWTICO": "oil_wti",
        "DGS10": "rate_10y",
        "FEDFUNDS": "rate_fedfunds",
        "T5YIE": "infl_breakeven_5y",
        "ICSA": "claims_initial",
        "IC4WSA": "claims_4wma",
        "DEXCAUS": "fx_cad_usd",
        "FRGSHPUSM649NCIS": "freight_cass",
        "WPU1171": "ppi_electronics",
        "MRTSSM443USS": "retail_elec_sales",
        "UMCSENT": "consumer_sentiment",
        "HOUST": "housing_starts",
        "DSPIC96": "real_disposable_income"
    }

    fred_w = (
        fred.where(F.col("series_id").isin(list(series_map.keys())))
            .withColumn("week_end_sun", week_ending_sunday(F.col("observation_date")))
            .groupBy("week_end_sun")
            .agg(
                *[
                    F.avg(F.when(F.col("series_id") == sid, F.col("value_double"))).alias(alias)
                    for sid, alias in series_map.items()
                ]
            )
            .withColumnRenamed("week_end_sun", "date")
    )

    trends = dlt.read("canada_business.silver.silver_google_trends")

    joined = fred_w.join(trends, on="date", how="left")

    # ---- Forward fill all macro columns ----
    window_spec = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)

    macro_cols = list(series_map.values())

    for c in macro_cols:
        joined = joined.withColumn(
            c,
            F.last(F.col(c), ignorenulls=True).over(window_spec)
        )

    return joined.withColumn("ingest_ts", F.current_timestamp())